Import Dependencies

In [ ]:
import random
from qdrant_client import QdrantClient

import instructor
import json
from pydantic import BaseModel, Field

from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client
import tiktoken

In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

Load all data from Qdrant

In [ ]:
all_points = []
next_offset = None
batch_size = 100

while True:
    points, next_offset = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01-hybrid-search",
        limit=batch_size,
        offset=next_offset,
        with_payload=True,
        with_vectors=False
    )
    all_points.extend(points)
    if next_offset is None:
        break

In [ ]:
len(all_points)

In [ ]:
all_points

Split the data into 2 groups: 50 items for synthetic question generation, 950 items to loop through and evaluate their relevance for previously generated synthetic questions.

In [ ]:
rng = random.Random(42)
indices = list(range(len(all_points)))

In [ ]:
rng.shuffle(indices)

In [ ]:
sampl_idx = set(indices[:50])

In [ ]:
sampl_idx

In [ ]:
sample_50 = [p for i, p in enumerate(all_points) if i in sampl_idx]

In [ ]:
remaining_points = [p for i, p in enumerate(all_points) if i not in sampl_idx]

In [ ]:
len(sample_50)

In [ ]:
len(remaining_points)

Generate synhetic questions

In [ ]:
all_context_sample_50 = [{"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in sample_50]

In [ ]:
all_context_sample_50